# Demo (Optional) - Orchestrated AI Agent in a Loop
**Day 1 - Session 1, Topic 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/demos-notebook/demo-agent-in-loop.ipynb)

**Goal:** Use an API-backed AI model to complete a coding task. Watch it make a mistake (violating an ownership boundary), and show how the orchestrated loop catches the violation and rejects it, just as it would for a human.

This requires an API key (e.g. `ANTHROPIC_API_KEY`) in your `.env` or environment. If none is provided, it safely falls back to a simulated diff to demonstrate the mechanics of the boundary check.


## 1. Setup

Locate the course files and put `demo_support` on the import path.


In [ ]:
# Setup: make the course files and demo_support importable.
# On Colab nothing is present yet, so clone the companion repo once.
# Locally this finds your existing checkout and clones nothing.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/agent-orchestration-companion.git"
MARKER = Path("lab-workspace-solution") / "router.py"


def find_repo_root():
    directory = Path.cwd()
    for _ in range(6):
        if (directory / MARKER).exists():
            return directory
        directory = directory.parent
    clone = Path.cwd() / "agent-orchestration-companion"
    if not (clone / MARKER).exists():
        print(f"Cloning {REPO_URL} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "-q", REPO_URL, str(clone)],
            check=True,
            env=dict(os.environ, GIT_TERMINAL_PROMPT="0"),
        )
    return clone


ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "day1" / "demos"))

# Colab has no global Git identity; the demos set a local one per sandbox repo.
print("Course root:", ROOT)
print("Git:", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())

## 2. Install the optional dependency

If you have an API key, we will use litellm. Run this block to install it.


In [ ]:
%pip install -q "litellm>=1.0.0" python-dotenv


## 3. Load the AI Agent

We use `llm_client` to route to whatever API key you provided. If no API key is detected, it falls back to a deterministic simulated diff so the orchestration logic still runs.


In [ ]:
def run_agent(root, prompt):
    try:
        from llm_client import get_completion, PROVIDER
        import os
        if not any(os.getenv(k) for k in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY", "AZURE_API_KEY", "AZURE_AD_TOKEN"]):
            return None, None
            
        sms_content = (root / "channels" / "sms.py").read_text()
        router_content = (root / "router.py").read_text()
        
        system = (
            "You are a coding agent. Return ONLY valid bash script content "
            "that uses EOF to rewrite files. Do NOT wrap the script in markdown code blocks like ```bash. "
            "Output RAW script only. Do NOT provide any explanations."
        )
        
        user = (
            f"Here is channels/sms.py:\n```python\n{sms_content}```\n\n"
            f"Here is router.py:\n```python\n{router_content}```\n\n"
            f"Task: {prompt}\n\n"
            "Output your answer as a bash script that writes the new contents to these files using cat << 'EOF' > path/to/file.py"
        )
        
        response = get_completion([
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ], tier="default", max_tokens=1000)
        
        script = response.replace("```bash\n", "").replace("```", "").strip()
        return script, f"{PROVIDER} API (default tier)"
    except Exception as e:
        print(f"Error calling LLM: {e}")
        return None, None

def simulate_agent(root):
    script = '''\
cat << 'EOF' > channels/push.py
def send_push(event):
    print("Push sent!")
EOF

cat << 'EOF' > router.py
from channels import email, sms, push

CHANNEL_ORDER = ("email", "sms", "push")

def route(event):
    pass
EOF
'''
    return script.strip(), "Simulated AI Agent (No API Key detected)"


## 4. Run the Task and Boundary Check

We assign the agent a task that tempts it to cross a boundary, then we use our orchestrator logic to catch it.


In [ ]:
from demo_support import assert_true, git, heading, sandbox, show_evidence

sandbox_context = sandbox()
WORK, BASE = sandbox_context.__enter__()

heading("AI Agent Task")
TASK = "We need a new push notification channel. Create channels/push.py and also register it in router.py's CHANNEL_ORDER."
show_evidence("Task", TASK)

SCRIPT, SOURCE = run_agent(WORK, TASK)
if not SCRIPT:
    SCRIPT, SOURCE = simulate_agent(WORK)
    
show_evidence("AI Engine", SOURCE)

import subprocess
subprocess.run(["bash", "-c", SCRIPT], cwd=WORK, check=False, capture_output=True)

git("add", "-A", cwd=WORK)
CHANGED = git("diff", "--name-only", "--cached", cwd=WORK).splitlines()

heading("Agent Modifications")
for f in CHANGED:
    show_evidence("Modified file", f)
    
heading("Orchestration Gate: Boundary Check")
OWNED_FILES = ["channels/push.py"]
show_evidence("Task Owned Files", ", ".join(OWNED_FILES))

VIOLATIONS = [f for f in CHANGED if f not in OWNED_FILES]
if VIOLATIONS:
    show_evidence("Gate Status", "REJECTED")
    show_evidence("Violations found", ", ".join(VIOLATIONS))
    SUCCESS = False
else:
    show_evidence("Gate Status", "PASSED")
    SUCCESS = True
    
heading("Evidence checks")
assert_true(len(CHANGED) > 0, "The agent modified the repository")
assert_true(not SUCCESS, "The orchestrator successfully caught the boundary violation")

sandbox_context.__exit__(None, None, None)
print("\nTakeaway: Real AI models will routinely ignore boundaries or attempt")
print("helpful overreach (like updating the router). Mechanical gates verify")
print("the output, not the AI's promises.")


### Expected output

- AI engine will report which model generated the script, or "Simulated AI Agent".
- The diff shows changes to `channels/push.py` and `router.py`.
- The boundary check gate compares this against the authorized `OWNED_FILES` list.
- It issues a `REJECTED` status because `router.py` was altered without authorization.
- Two `[verified]` checks pass, proving the orchestration gate works on agentic code.
